# 256K Long-Context Compliance Audit & Multi-Attribute RLAIF Studio with Nemotron-3 Ultra (550B/55B MoE)

> **Google Cloud | Gemini Enterprise Agent Platform | NVIDIA Nemotron-3 Ultra in Model Garden**

---

### 📖 Executive Overview
**NVIDIA Nemotron-3 Ultra** represents the frontier flagship of the Nemotron family, featuring a massive **550B Total / 55B Active MoE** architecture with a **256K Context Window** (`262,144` tokens), speculative decoding, and native NVFP4 acceleration on NVIDIA B200 / H200 clusters.

This notebook demonstrates two flagship enterprise workloads:
1. **Part 1: 256K Long-Context Compliance Audit**: Ingests multi-service repository codebases alongside strict regulatory frameworks (HIPAA Security Rule, SOC 2 Type II, GDPR Article 32) to detect security vulnerabilities and synthesize patch diffs.
2. **Part 2: Multi-Attribute RLAIF Quality Scoring Studio**: Dual-turn synthetic data generation and multi-attribute reward scoring across 5 distinct dimensions (*Helpfulness, Truthfulness, Conciseness, Compliance, Complexity*).

> [!TIP]
> **Flexible Endpoint Operation Modes:**
> * **Mode 1 (`AUTO_DISCOVER`) [Default]**: Automatically scans and binds to active Model Garden deployments in your GCP project. (To deploy via UI ahead of time: [Vertex AI Model Garden](https://console.cloud.google.com/vertex-ai/model-garden), recommended profile: **a4-highgpu-8g or a3-ultragpu-8g**).
> * **Mode 2 (`USE_EXISTING_ENDPOINT`)**: Bind directly to any custom endpoint or fine-tuned model by setting `CUSTOM_ENDPOINT_ID`.
> * **Mode 3 (`CREATE_CUSTOM_ENDPOINT`)**: Programmatically create a new Vertex AI Endpoint and deploy your custom container or model weights directly from the notebook.

---

### 📋 Prerequisites & Setup
* Target Model: **Nemotron-3 Ultra** (`nemotron-3-ultra-550b-a55b-nvfp4` or `bf16`) or custom endpoint.
* Recommended Accelerator: `a4-highgpu-8g` (8x NVIDIA B200 1536GB) or `a3-ultragpu-8g` (8x NVIDIA H200 1128GB).


### Step 1: Harmonized Dependency Installation


In [ ]:
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

import sys
import subprocess

# Harmonized dependency installation with conflict prevention
!pip install --quiet --no-warn-conflicts "google-cloud-aiplatform>=1.70.0" "openai>=1.50.0,<2.0.0" "pydantic>=2.0.0,<3.0.0" "rich>=13.7.0,<14.0.0" "requests>=2.31.0,<=2.32.4" "protobuf>=3.20.2,<5.0.0dev"

print("✓ Harmonized dependencies installed successfully.")


### Step 2: Environment Configuration & Endpoint Operation Selector

> [!TIP]
> **Flexible Endpoint Operation Modes:**
> * **Mode 1 (`AUTO_DISCOVER`) [Default]**: Automatically scans and binds to active Model Garden deployments in your GCP project. (To deploy via UI ahead of time: [Vertex AI Model Garden](https://console.cloud.google.com/vertex-ai/model-garden), recommended profile: **a4-highgpu-8g or a3-ultragpu-8g**).
> * **Mode 2 (`USE_EXISTING_ENDPOINT`)**: Bind directly to any custom endpoint or fine-tuned model by setting `CUSTOM_ENDPOINT_ID`.
> * **Mode 3 (`CREATE_CUSTOM_ENDPOINT`)**: Programmatically create a new Vertex AI Endpoint and deploy your custom container or model weights directly from the notebook.


In [ ]:
import os
import subprocess
from google.cloud import aiplatform
from rich.console import Console
from rich.table import Table
from rich.panel import Panel

console = Console()

# ==============================================================================
# Step 2: Environment Configuration & Endpoint Operation Selector
# ==============================================================================
# Choose your connection/deployment operation:
# 1. "AUTO_DISCOVER" : (Default) Auto-detects and binds to active Model Garden endpoints.
# 2. "USE_EXISTING_ENDPOINT": Binds directly to your custom or existing endpoint ID/Name.
# 3. "CREATE_CUSTOM_ENDPOINT": Programmatically creates a new Vertex AI endpoint and
#                              deploys a custom container/model artifact.
# ==============================================================================

PROJECT_ID = ""                # @param {type:"string"} - Set your GCP Project ID (Leave blank to auto-detect)
REGION = "us-central1"         # @param ["us-central1", "us-east4", "us-west1", "europe-west4"] {allow-input: true}
OPERATION_MODE = "AUTO_DISCOVER" # @param ["AUTO_DISCOVER", "USE_EXISTING_ENDPOINT", "CREATE_CUSTOM_ENDPOINT"]

# --- Mode: USE_EXISTING_ENDPOINT Settings ---
CUSTOM_ENDPOINT_ID = ""        # @param {type:"string"} - e.g. "1234567890" or "projects/.../endpoints/..."

# --- Mode: CREATE_CUSTOM_ENDPOINT Settings ---
CUSTOM_ENDPOINT_DISPLAY_NAME = "ultra-custom-endpoint" # @param {type:"string"}
CUSTOM_SERVING_CONTAINER_URI = "us-docker.pkg.dev/vertex-ai/vertex-vision-model-garden-dockers/vllm-serve:latest" # @param {type:"string"}
CUSTOM_ARTIFACT_URI = ""       # @param {type:"string"} - Optional: Cloud Storage path (e.g. gs://your-bucket/model-weights)
CUSTOM_MACHINE_TYPE = "a4-highgpu-8g" # @param ["g4-standard-48", "g4-standard-384", "g2-standard-16", "g2-standard-96", "a4-highgpu-8g", "a3-ultragpu-8g"] {allow-input: true}
CUSTOM_ACCELERATOR_TYPE = "NVIDIA_B200" # @param ["NVIDIA_RTX_PRO_6000", "NVIDIA_L4", "NVIDIA_B200", "NVIDIA_H200"] {allow-input: true}
CUSTOM_ACCELERATOR_COUNT = 8   # @param {type:"integer"}

# 1. Resolve GCP Project ID
if not PROJECT_ID.strip():
    try:
        PROJECT_ID = subprocess.check_output(
            ["gcloud", "config", "get-value", "project"], 
            stderr=subprocess.DEVNULL
        ).decode().strip()
    except Exception:
        PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT", "cpe-slarbi-nvd-ant-demos")

console.print(f"[bold green]✓ GCP Project:[/bold green] [cyan]{PROJECT_ID}[/cyan] | [bold green]Region:[/bold green] [cyan]{REGION}[/cyan] | [bold green]Mode:[/bold green] [yellow]{OPERATION_MODE}[/yellow]")
aiplatform.init(project=PROJECT_ID, location=REGION)

# 2. Unified Endpoint Resolver & Deployer
def resolve_or_create_endpoint(
    project_id: str, 
    location: str, 
    target_keywords: list,
    mode: str = "AUTO_DISCOVER",
    custom_endpoint_id: str = "",
    create_params: dict = None
) -> aiplatform.Endpoint:
    # -------------------------------------------------------------------------
    # Path 1: Connect to an Existing Custom Endpoint
    # -------------------------------------------------------------------------
    if mode == "USE_EXISTING_ENDPOINT" or (custom_endpoint_id and custom_endpoint_id.strip()):
        ep_name = custom_endpoint_id.strip()
        if not ep_name:
            raise ValueError("OPERATION_MODE is 'USE_EXISTING_ENDPOINT', but CUSTOM_ENDPOINT_ID is empty.")
        
        ep_path = ep_name if ep_name.startswith("projects/") else f"projects/{project_id}/locations/{location}/endpoints/{ep_name}"
        ep = aiplatform.Endpoint(ep_path)
        console.print(Panel(
            f"[bold]Display Name:[/bold] {ep.display_name}\n"
            f"[bold]Endpoint ID:[/bold]  {ep.name.split('/')[-1]}\n"
            f"[bold]Resource:[/bold]     {ep.name}",
            title="✓ BOUND TO CUSTOM ENDPOINT",
            border_style="green"
        ))
        return ep

    # -------------------------------------------------------------------------
    # Path 2: Create & Deploy a Custom Endpoint on Demand
    # -------------------------------------------------------------------------
    if mode == "CREATE_CUSTOM_ENDPOINT":
        p = create_params or {}
        disp_name = p.get("display_name", "custom-nemotron-endpoint")
        container_uri = p.get("container_uri", "us-docker.pkg.dev/vertex-ai/vertex-vision-model-garden-dockers/vllm-serve:latest")
        artifact_uri = p.get("artifact_uri", "")
        mach_type = p.get("machine_type", "a4-highgpu-8g")
        acc_t = p.get("accelerator_type", "NVIDIA_B200")
        acc_c = p.get("accelerator_count", 8)

        console.print(Panel(
            f"[bold]Endpoint Name:[/bold]     {disp_name}\n"
            f"[bold]Serving Container:[/bold] {container_uri}\n"
            f"[bold]Hardware Profile:[/bold]  {mach_type} ({acc_c}x {acc_t})",
            title="🚀 INITIATING CUSTOM ENDPOINT DEPLOYMENT",
            border_style="yellow"
        ))

        console.print("⏳ [1/3] Registering custom model with Vertex AI...")
        upload_kwargs = {
            "display_name": f"{disp_name}-model",
            "serving_container_image_uri": container_uri,
        }
        if artifact_uri.strip():
            upload_kwargs["artifact_uri"] = artifact_uri.strip()

        model_res = aiplatform.Model.upload(**upload_kwargs)
        console.print(f"✓ Model registered: [cyan]{model_res.resource_name}[/cyan]")

        console.print("⏳ [2/3] Creating dedicated Vertex AI endpoint...")
        custom_endpoint = aiplatform.Endpoint.create(display_name=disp_name)
        console.print(f"✓ Endpoint created: [cyan]{custom_endpoint.resource_name}[/cyan]")

        console.print("⏳ [3/3] Deploying model to endpoint (provisioning compute and loading weights)...")
        model_res.deploy(
            endpoint=custom_endpoint,
            machine_type=mach_type,
            accelerator_type=acc_t,
            accelerator_count=acc_c,
            traffic_percentage=100,
            sync=True
        )
        console.print(Panel(
            f"[bold]Display Name:[/bold] {custom_endpoint.display_name}\n"
            f"[bold]Endpoint ID:[/bold]  {custom_endpoint.name.split('/')[-1]}\n"
            f"[bold]Resource:[/bold]     {custom_endpoint.name}",
            title="✓ CUSTOM ENDPOINT DEPLOYED SUCCESSFULLY",
            border_style="bold green"
        ))
        return custom_endpoint

    # -------------------------------------------------------------------------
    # Path 3: Auto-Discovery of Active Model Garden Endpoints (Default)
    # -------------------------------------------------------------------------
    console.print(f"🔍 Scanning for active Vertex AI endpoints in [cyan]{project_id}[/cyan] ({location})...")
    endpoints = aiplatform.Endpoint.list(order_by="create_time desc")
    
    if not endpoints:
        console.print(Panel(
            f"No active endpoints found in project [cyan]{project_id}[/cyan] / [cyan]{location}[/cyan].\n\n"
            f"1. Open Model Garden: https://console.cloud.google.com/vertex-ai/model-garden\n"
            f"2. Or set OPERATION_MODE = 'CREATE_CUSTOM_ENDPOINT' to deploy directly.\n"
            f"3. Or set OPERATION_MODE = 'USE_EXISTING_ENDPOINT' with CUSTOM_ENDPOINT_ID.",
            title="⚠️ NO ACTIVE ENDPOINTS FOUND",
            border_style="bold red"
        ))
        raise RuntimeError("No active endpoints found. Please deploy a Model Garden model or select a custom mode.")

    table = Table(title=f"Active Endpoints in {project_id}", border_style="blue")
    table.add_column("#", style="dim", width=4)
    table.add_column("Display Name", style="bold white")
    table.add_column("Endpoint ID", style="cyan")
    
    for idx, ep in enumerate(endpoints, start=1):
        table.add_row(str(idx), ep.display_name, ep.name.split("/")[-1])
    console.print(table)

    # 1st Priority: Match target model keywords
    matching = [
        ep for ep in endpoints 
        if any(k.lower() in (ep.display_name or "").lower() for k in target_keywords)
    ]

    if matching:
        selected = matching[0]
        console.print(Panel(
            f"[bold]Attached Model:[/bold]   {selected.display_name}\n"
            f"[bold]Endpoint ID:[/bold]      {selected.name.split('/')[-1]}\n"
            f"[bold]Resource Path:[/bold]    {selected.name}",
            title=f"✓ AUTO-ATTACHED: Nemotron-3 Ultra (550B/55B)",
            border_style="bold green"
        ))
        return selected

    # 2nd Priority: Fallback to any active Nemotron / NVIDIA endpoint
    generic_nemotron = [
        ep for ep in endpoints 
        if any(k in (ep.display_name or "").lower() for k in ["nemotron", "nvidia"])
    ]
    if generic_nemotron:
        selected = generic_nemotron[0]
        console.print(Panel(
            f"[bold]Attached Model:[/bold]   {selected.display_name}\n"
            f"[bold]Endpoint ID:[/bold]      {selected.name.split('/')[-1]}\n"
            f"[bold]Resource Path:[/bold]    {selected.name}",
            title="💡 AUTO-ATTACHED TO ACTIVE NEMOTRON ENDPOINT",
            border_style="bold yellow"
        ))
        return selected

    # 3rd Priority: Fallback to first available active endpoint
    selected = endpoints[0]
    console.print(f"💡 [dim]Fallback: Binding to active endpoint '{selected.display_name}' (ID: {selected.name.split('/')[-1]}).[/dim]")
    return selected

target_endpoint = resolve_or_create_endpoint(
    PROJECT_ID, 
    REGION, 
    target_keywords=["ultra", "nemotron-3-ultra", "nemotron", "nvidia"],
    mode=OPERATION_MODE,
    custom_endpoint_id=CUSTOM_ENDPOINT_ID,
    create_params={
        "display_name": CUSTOM_ENDPOINT_DISPLAY_NAME,
        "container_uri": CUSTOM_SERVING_CONTAINER_URI,
        "artifact_uri": CUSTOM_ARTIFACT_URI,
        "machine_type": CUSTOM_MACHINE_TYPE,
        "accelerator_type": CUSTOM_ACCELERATOR_TYPE,
        "accelerator_count": CUSTOM_ACCELERATOR_COUNT
    }
)


### Step 3: Part 1 - Ingest Codebase & Regulatory Framework (HIPAA / SOC 2)


In [ ]:
CODEBASE_CONTEXT = """
// File: services/patient_records/controller.py
from flask import Blueprint, request, jsonify
import logging

patient_bp = Blueprint('patients', __name__)
logger = logging.getLogger(__name__)

@patient_bp.route('/v1/patients/export', methods=['POST'])
def export_patient_phi():
    patient_id = request.json.get('patient_id')
    ssn = request.json.get('ssn')
    diagnosis = request.json.get('diagnosis_records')
    
    # Non-compliant: Unencrypted PHI printed directly to standard log stream
    logger.info(f"Exporting PHI for Patient ID: {patient_id} | SSN: {ssn} | Records: {diagnosis}")
    
    # Non-compliant: Backup exported to unauthenticated public Cloud Storage bucket
    backup_url = f"https://storage.googleapis.com/public-health-backups/{patient_id}.json"
    return jsonify({"status": "exported", "download_url": backup_url}), 200
"""

REGULATORY_SPEC = """
HIPAA Security Rule 45 CFR § 164.312:
1. Technical Safeguards: Implement mechanism to encrypt and decrypt electronic protected health information (ePHI) in transit and at rest.
2. Audit Controls: Record and examine activity in information systems containing ePHI without logging raw unmasked ePHI / SSN.
3. Access Control: Restrict access to ePHI to authorized persons only (No public bucket storage).
"""

console.print(Panel(
    "Ingested Patient Records Controller & HIPAA Security Rule (45 CFR § 164.312)",
    title="✓ Codebase Context & Regulatory Compliance Framework Loaded",
    border_style="cyan"
))


### Step 4: Execute Long-Context Compliance Audit & Patch Diff Synthesis (Formatted View & Max Tokens)
Nemotron-3 Ultra analyzes the full context and generates unified git patch diffs rendered in clean Markdown with generous token budget (`max_tokens=2048`).


In [ ]:
import json
import re

def extract_clean_content(prediction_obj) -> str:
    """
    Extracts purely the clean assistant text from any Vertex AI / vLLM / OpenAI response format,
    filtering out raw API dictionaries, metadata envelopes, usage stats, and unicode artifacts.
    """
    if isinstance(prediction_obj, list) and len(prediction_obj) > 0:
        prediction_obj = prediction_obj[0]

    if isinstance(prediction_obj, str):
        try:
            prediction_obj = json.loads(prediction_obj)
        except Exception:
            pass

    content = ""
    if isinstance(prediction_obj, dict):
        # 1. Check for OpenAI/vLLM 'choices' format
        choices = prediction_obj.get("choices", [])
        if isinstance(choices, list) and len(choices) > 0:
            first_choice = choices[0]
            if isinstance(first_choice, dict):
                msg = first_choice.get("message", {})
                if isinstance(msg, dict):
                    content = msg.get("content", "")
                    if not content and "reasoning" in msg:
                        content = msg.get("reasoning", "")
                    if not content and "reasoning_content" in msg:
                        content = msg.get("reasoning_content", "")
                elif isinstance(msg, str):
                    content = msg
                if not content:
                    content = first_choice.get("text", "")
            elif isinstance(first_choice, str):
                content = first_choice

        # 2. Check for standard 'content', 'text', or 'predictions'
        if not content:
            content = prediction_obj.get("content", prediction_obj.get("text", ""))

        # 3. Direct message dict check
        if not content and "role" in prediction_obj and "content" in prediction_obj:
            content = prediction_obj.get("content", "")

    elif isinstance(prediction_obj, str):
        content = prediction_obj

    if not content:
        content = str(prediction_obj)

    # Normalize unicode spacing artifacts ( ,  )
    cleaned = str(content).replace("\u202f", " ").replace("\u00a0", " ").strip()
    return cleaned

def extract_json_from_text(raw_text: str) -> dict:
    """
    Robustly extracts and parses a JSON dictionary from LLM output,
    handling markdown blocks, preambles, reasoning wrappers, and trailing commas.
    """
    text = raw_text.strip()
    try:
        return json.loads(text)
    except Exception:
        pass

    if "```json" in text:
        content = text.split("```json", 1)[1]
        if "```" in content:
            content = content.split("```", 1)[0]
        try:
            return json.loads(content.strip())
        except Exception:
            sanitized = re.sub(r',\s*([\}\]])', r'\1', content.strip())
            try:
                return json.loads(sanitized)
            except Exception:
                pass

    if "```" in text:
        content = text.split("```", 1)[1]
        if "```" in content:
            content = content.split("```", 1)[0]
        try:
            return json.loads(content.strip())
        except Exception:
            sanitized = re.sub(r',\s*([\}\]])', r'\1', content.strip())
            try:
                return json.loads(sanitized)
            except Exception:
                pass

    first_brace = text.find("{")
    last_brace = text.rfind("}")
    if first_brace != -1 and last_brace != -1 and last_brace > first_brace:
        candidate = text[first_brace:last_brace + 1]
        try:
            return json.loads(candidate)
        except Exception:
            sanitized = re.sub(r',\s*([\}\]])', r'\1', candidate)
            try:
                return json.loads(sanitized)
            except Exception:
                pass

    raise ValueError(f"No valid JSON object found in model output: {text[:200]}")

from IPython.display import display, Markdown

AUDIT_PROMPT = f"""You are a Principal Enterprise Compliance Officer & Security Auditor powered by NVIDIA Nemotron-3 Ultra.
Perform a thorough compliance audit of the provided repository against the regulatory specification.

CODEBASE:
{CODEBASE_CONTEXT}

REGULATORY SPECIFICATION:
{REGULATORY_SPEC}

OUTPUT FORMAT:
### 1. 🛡️ Regulatory Violation Matrix
| File & Line | Regulatory Section | Severity | Violation Details |
|---|---|---|---|
| ... | ... | ... | ... |

### 2. 📝 Remediation Git Patch Diff
```diff
[Provide exact unified diff]
```
"""

payload = {
    "instances": [
        {
            "@requestFormat": "chatCompletions",
            "messages": [{"role": "user", "content": AUDIT_PROMPT}],
            "max_tokens": 2048,
            "temperature": 0.1
        }
    ]
}

try:
    res = target_endpoint.predict(instances=payload["instances"])
    audit_output = extract_clean_content(res.predictions)
except Exception:
    res = target_endpoint.predict(instances=[{"prompt": AUDIT_PROMPT, "max_tokens": 2048}])
    audit_output = extract_clean_content(res.predictions)

try:
    display(Markdown(f"### 📋 HIPAA / SOC 2 Compliance Audit & Patch Synthesis\n\n{audit_output}"))
except Exception:
    console.print(Panel(audit_output, title="📋 REGULATORY AUDIT & PATCH SYNTHESIS", border_style="bold green"))


### Step 5: Part 2 - Multi-Attribute RLAIF Quality Scoring Studio (Scorecard Dashboard & Max Tokens)
Demonstrates multi-attribute evaluation across 5 reward metrics: *Helpfulness, Truthfulness, Conciseness, Compliance, and Complexity* rendered in a visual scorecard table with `max_tokens=1024`.


In [ ]:
import json
from pydantic import BaseModel, Field

class RLAIFScorecard(BaseModel):
    helpfulness_score: float = Field(ge=1.0, le=10.0, description="Clarity and direct usefulness (1-10)")
    truthfulness_score: float = Field(ge=1.0, le=10.0, description="Factual accuracy and grounding (1-10)")
    conciseness_score: float = Field(ge=1.0, le=10.0, description="Information density without fluff (1-10)")
    compliance_score: float = Field(ge=1.0, le=10.0, description="Adherence to security & privacy guidelines (1-10)")
    complexity_score: float = Field(ge=1.0, le=10.0, description="Depth of technical solution (1-10)")
    composite_grade: str = Field(description="A+, A, B, C, or REJECT")
    critique_summary: str = Field(description="Detailed evaluation rationale")

schema_str = json.dumps(RLAIFScorecard.model_json_schema(), indent=2)

EVAL_PROMPT = f"""You are an AI Alignment Judge powered by Nemotron-3 Ultra.
Evaluate the following generated remediation patch across 5 attributes and output a valid JSON object strictly matching this schema:
{schema_str}

AUDIT & PATCH CANDIDATE:
{audit_output}

OUTPUT ONLY VALID JSON:"""

console.print("Evaluating patch candidate with Multi-Attribute RLAIF Judge...")

try:
    res_eval = target_endpoint.predict(instances=[{
        "@requestFormat": "chatCompletions",
        "messages": [{"role": "user", "content": EVAL_PROMPT}],
        "max_tokens": 1024,
        "temperature": 0.0
    }])
    eval_text = extract_clean_content(res_eval.predictions)
except Exception:
    res_eval = target_endpoint.predict(instances=[{"prompt": EVAL_PROMPT, "max_tokens": 1024}])
    eval_text = extract_clean_content(res_eval.predictions)

try:
    scorecard_dict = extract_json_from_text(eval_text)
    scorecard = RLAIFScorecard.model_validate(scorecard_dict)
    
    rlaif_table = Table(title="🎯 Multi-Attribute RLAIF Quality Scorecard", border_style="magenta")
    rlaif_table.add_column("Attribute Dimension", style="bold cyan")
    rlaif_table.add_column("Score (out of 10)", style="bold white")
    rlaif_table.add_column("Rating Bar", style="green")
    
    metrics = [
        ("Helpfulness", scorecard.helpfulness_score),
        ("Truthfulness", scorecard.truthfulness_score),
        ("Conciseness", scorecard.conciseness_score),
        ("Compliance", scorecard.compliance_score),
        ("Complexity", scorecard.complexity_score)
    ]
    
    for name, score in metrics:
        bar = "█" * int(score) + "░" * (10 - int(score))
        rlaif_table.add_row(name, f"{score:.1f}/10", bar)
        
    console.print(rlaif_table)
    console.print(Panel(
        f"[bold]Composite Grade:[/bold]   [bold green]{scorecard.composite_grade}[/bold green]\n"
        f"[bold]Critique Summary:[/bold]  {scorecard.critique_summary}",
        title="ALIGNMENT VERDICT",
        border_style="bold green"
    ))
except Exception as e:
    console.print(f"[red]Evaluation parse error:[/red] {e}")
    console.print(Panel(eval_text, title="Raw Model Output", border_style="yellow"))


### Step 6: Safe Teardown & Endpoint Lifecycle Management


In [ ]:
DELETE_SCRATCH_ENDPOINT = False  # @param {type:"boolean"}
if DELETE_SCRATCH_ENDPOINT and 'target_endpoint' in locals():
    console.print(f"[bold red]Cleaning up endpoint:[/bold red] {target_endpoint.display_name}...")
    target_endpoint.delete(force=True)
    console.print("[bold green]✓ Cleanup completed.[/bold green]")
else:
    console.print("[dim]Preserving endpoint for subsequent sessions.[/dim]")
